# DistilBERT + LoRA: fixed hyperparameter search

Честный эксперимент для сравнения с full fine-tuning. Все конфигурации обучаются только на фиксированном train split, лучший checkpoint выбирается только по validation macro-F1, а test split используется ровно один раз для финальной оценки победителя.

Fixed grid намеренно мал для запуска на Google Colab T4: перебираются два learning rate и два LoRA rank; остальные параметры зафиксированы одинаково во всех запусках.


In [ ]:
%cd /content
!rm -rf LORA-course-project
!git clone https://github.com/NikitaBaranenkov/LORA-course-project.git
%cd LORA-course-project
!ls
!ls src

In [ ]:
%pip uninstall -y torchao
%pip install -q --upgrade peft transformers accelerate datasets evaluate scikit-learn

In [ ]:
import gc
import inspect
import json
import random
import time
from itertools import product

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

from src.constants import (
    BASE_MODEL_DISTILBERT,
    DATASET_NAME,
    FINAL_TEST_EVALUATION_ONLY,
    MAX_LENGTH,
    PROJECT_ROOT,
    REPORTS_TABLES_DIR,
    SEED,
    SELECTION_METRIC,
    TEST_CSV_PATH,
    TRAIN_CSV_PATH,
    VALIDATION_CSV_PATH,
    id2label,
    label2id,
)

RUN_NAME = "distilbert_lora_hparam_search"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / RUN_NAME
CHECKPOINTS_DIR = ARTIFACTS_DIR / "checkpoints"
BEST_ADAPTER_DIR = ARTIFACTS_DIR / "best_adapter"
RUN_TABLES_DIR = REPORTS_TABLES_DIR / "distilbert_lora"

SWEEP_RESULTS_PATH = RUN_TABLES_DIR / "distilbert_lora_hparam_search_validation_results.csv"
FINAL_TEST_RESULTS_PATH = RUN_TABLES_DIR / "distilbert_lora_best_hparam_test_results.csv"
EXPERIMENT_CONFIG_PATH = RUN_TABLES_DIR / "distilbert_lora_hparam_search_config.json"


## Импорты и настройка окружения

## Загрузка фиксированных разбиений данных

In [ ]:
train_df = pd.read_csv(TRAIN_CSV_PATH).reset_index(drop=True)
validation_df = pd.read_csv(VALIDATION_CSV_PATH).reset_index(drop=True)
test_df = pd.read_csv(TEST_CSV_PATH).reset_index(drop=True)

for split_df in (train_df, validation_df, test_df):
    split_df["label"] = split_df["label"].astype(int)

raw_dataframes = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

raw_datasets = DatasetDict({
    split_name: Dataset.from_pandas(split_df, preserve_index=False)
    for split_name, split_df in raw_dataframes.items()
})
raw_datasets = raw_datasets.rename_column("label", "labels")

raw_datasets

## Токенизация текста

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DISTILBERT)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label_name"],
)
tokenized_datasets.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

sample = tokenized_datasets["train"][0]
print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["labels"].item())

## Метрики качества

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
    }


def softmax_numpy(logits):
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
    probabilities = np.exp(shifted_logits)
    return probabilities / np.sum(probabilities, axis=1, keepdims=True)


def main_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }


assert SELECTION_METRIC == "macro_f1"
assert SELECTION_METRIC in compute_metrics((np.zeros((1, len(id2label))), np.array([0])))

## Фабрика моделей DistilBERT + LoRA

Каждая конфигурация sweep получает заново инициализированную базовую модель при одном и том же seed. Классификационная голова сохраняется и обучается вместе с LoRA-адаптерами, как в исходном ноутбуке.


In [ ]:
NUM_LABELS = len(id2label)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def new_base_model():
    return AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_DISTILBERT,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
    )


set_global_seed(SEED)
probe_base_model = new_base_model()
head_modules_to_save = [
    module_name
    for module_name in ("pre_classifier", "classifier")
    if hasattr(probe_base_model, module_name)
]
assert head_modules_to_save == ["pre_classifier", "classifier"], (
    "Expected DistilBERT classification head modules were not found: "
    f"{head_modules_to_save}"
)
del probe_base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Device:", device)
print("Base model:", BASE_MODEL_DISTILBERT)
print("DistilBERT classification head modules to train and save:", head_modules_to_save)


## Fixed grid для LoRA

Для PEFT наиболее содержательны скорость обучения и ёмкость адаптера. Поэтому проверяются `learning_rate` из {2e-4, 5e-4} и `r` из {8, 16}; `lora_alpha=2*r` сохраняет одинаковое масштабирование обновления. Dropout, budget эпох, batch size и regularization фиксированы, чтобы сетка оставалась выполнимой на T4.


In [ ]:
LORA_GRID = [
    {
        "learning_rate": learning_rate,
        "r": rank,
        "lora_alpha": 2 * rank,
        "lora_dropout": 0.1,
        "num_train_epochs": 4,
        "weight_decay": 0.01,
        "warmup_ratio": 0.06,
    }
    for learning_rate, rank in product([2e-4, 5e-4], [8, 16])
]

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LOGGING_STEPS = 50


def build_lora_model(config):
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=config["r"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        bias="none",
        target_modules=["q_lin", "v_lin"],
        modules_to_save=list(head_modules_to_save),
    )
    return get_peft_model(new_base_model(), lora_config)


def summarize_parameters(model, print_summary=False):
    total_params = sum(parameter.numel() for parameter in model.parameters())
    trainable_names = [
        name for name, parameter in model.named_parameters() if parameter.requires_grad
    ]
    trainable_params = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    trainable_share = trainable_params / total_params

    if print_summary:
        print(f"Total parameters:     {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Trainable share:      {100 * trainable_share:.4f}%")

    assert any("lora_" in name for name in trainable_names), "No trainable LoRA parameters found."
    for head_module in head_modules_to_save:
        assert any(f".{head_module}." in name for name in trainable_names), (
            f"The classification head module {head_module!r} is not trainable."
        )
    unexpected_trainable_names = [
        name
        for name in trainable_names
        if "lora_" not in name
        and not any(f".{head_module}." in name for head_module in head_modules_to_save)
    ]
    assert not unexpected_trainable_names, (
        "Unexpected trainable backbone parameters: "
        f"{unexpected_trainable_names[:10]}"
    )
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "trainable_share": trainable_share,
    }


print("Number of LoRA configurations:", len(LORA_GRID))
pd.DataFrame(LORA_GRID)


### Проверка forward pass до sweep


In [ ]:
set_global_seed(SEED)
smoke_model = build_lora_model(LORA_GRID[0]).to(device)
summarize_parameters(smoke_model, print_summary=True)
smoke_model.eval()

smoke_batch_size = min(2, len(tokenized_datasets["validation"]))
smoke_batch = tokenized_datasets["validation"][:smoke_batch_size]
smoke_inputs = {
    key: smoke_batch[key].to(device)
    for key in ("input_ids", "attention_mask")
}
with torch.no_grad():
    smoke_output = smoke_model(**smoke_inputs)

expected_logits_shape = (smoke_batch_size, NUM_LABELS)
assert tuple(smoke_output.logits.shape) == expected_logits_shape
print("Forward pass logits shape:", tuple(smoke_output.logits.shape))

del smoke_model, smoke_output
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Hyperparameter search только по validation macro-F1

В этом блоке `test` не используется. Для каждого набора параметров `Trainer` сохраняет лучший epoch по validation macro-F1; затем среди четырёх запусков выбирается checkpoint с наибольшим `val_macro_f1`.


In [ ]:
assert SELECTION_METRIC == "macro_f1"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_TABLES_DIR.mkdir(parents=True, exist_ok=True)


def make_training_arguments(config_id, config):
    training_args_kwargs = {
        "output_dir": str(CHECKPOINTS_DIR / f"config_{config_id:02d}"),
        "overwrite_output_dir": True,
        "learning_rate": config["learning_rate"],
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "num_train_epochs": config["num_train_epochs"],
        "weight_decay": config["weight_decay"],
        "warmup_ratio": config["warmup_ratio"],
        "logging_steps": LOGGING_STEPS,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": SELECTION_METRIC,
        "greater_is_better": True,
        "save_total_limit": 1,
        "report_to": "none",
        "seed": SEED,
        "data_seed": SEED,
        "fp16": torch.cuda.is_available(),
    }
    signature = inspect.signature(TrainingArguments.__init__)
    strategy_key = "eval_strategy" if "eval_strategy" in signature.parameters else "evaluation_strategy"
    training_args_kwargs[strategy_key] = "epoch"
    return TrainingArguments(**training_args_kwargs)


sweep_records = []
best_run = None
for config_id, config in enumerate(LORA_GRID, start=1):
    print(f"\nTraining LoRA configuration {config_id}/{len(LORA_GRID)}:", config)
    set_global_seed(SEED)
    run_model = build_lora_model(config).to(device)
    parameter_summary = summarize_parameters(run_model)
    run_trainer = Trainer(
        model=run_model,
        args=make_training_arguments(config_id, config),
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    start_time = time.perf_counter()
    run_trainer.train()
    run_training_time_sec = time.perf_counter() - start_time
    validation_metrics = run_trainer.evaluate(
        tokenized_datasets["validation"],
        metric_key_prefix="validation",
    )
    assert run_trainer.state.best_model_checkpoint is not None

    record = {
        "config_id": config_id,
        **config,
        "val_macro_f1": float(validation_metrics["validation_macro_f1"]),
        "val_weighted_f1": float(validation_metrics["validation_weighted_f1"]),
        "val_accuracy": float(validation_metrics["validation_accuracy"]),
        "training_time_sec": run_training_time_sec,
        "best_checkpoint": run_trainer.state.best_model_checkpoint,
        **parameter_summary,
    }
    sweep_records.append(record)
    if best_run is None or record["val_macro_f1"] > best_run["record"]["val_macro_f1"]:
        best_run = {
            "record": dict(record),
            "config": dict(config),
            "parameter_summary": dict(parameter_summary),
        }

    del run_trainer, run_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

sweep_results_df = (
    pd.DataFrame(sweep_records)
    .sort_values("val_macro_f1", ascending=False)
    .reset_index(drop=True)
)
sweep_results_df.to_csv(SWEEP_RESULTS_PATH, index=False)

best_config = best_run["config"]
selected_training_time_sec = best_run["record"]["training_time_sec"]
print("\nBest LoRA configuration selected only on validation macro-F1:")
print(best_config)
print("Best checkpoint:", best_run["record"]["best_checkpoint"])
sweep_results_df


## Единственная финальная оценка на test split

Checkpoint победившей validation-конфигурации загружается заново. Только после завершения выбора модели выполняется один вызов `predict` для `test`.


In [ ]:
assert FINAL_TEST_EVALUATION_ONLY is True
assert best_run is not None

set_global_seed(SEED)
best_base_model = new_base_model()
best_model = PeftModel.from_pretrained(
    best_base_model,
    best_run["record"]["best_checkpoint"],
    is_trainable=False,
).to(device)

final_eval_args = TrainingArguments(
    output_dir=str(ARTIFACTS_DIR / "final_evaluation"),
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    report_to="none",
    fp16=torch.cuda.is_available(),
)
best_trainer = Trainer(model=best_model, args=final_eval_args, compute_metrics=compute_metrics)

# This is intentionally the only test-set inference in the notebook.
test_prediction_output = best_trainer.predict(
    tokenized_datasets["test"],
    metric_key_prefix="test",
)
test_logits = test_prediction_output.predictions
test_probabilities = softmax_numpy(test_logits)
test_predictions = np.argmax(test_probabilities, axis=1)
test_labels = raw_dataframes["test"]["label"].to_numpy()
test_metrics = main_metrics(test_labels, test_predictions)

class_names = [id2label[class_id] for class_id in sorted(id2label)]
test_classification_report = classification_report(
    test_labels,
    test_predictions,
    labels=sorted(id2label),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
test_classification_report_df = (
    pd.DataFrame(test_classification_report).transpose().rename_axis("class").reset_index()
)
test_confusion_matrix_df = pd.DataFrame(
    confusion_matrix(test_labels, test_predictions, labels=sorted(id2label)),
    index=[f"true_{class_name}" for class_name in class_names],
    columns=[f"pred_{class_name}" for class_name in class_names],
)

test_predictions_df = raw_dataframes["test"].copy()
test_predictions_df["true_label"] = test_labels
test_predictions_df["true_label_name"] = test_predictions_df["true_label"].map(id2label)
test_predictions_df["pred_label"] = test_predictions
test_predictions_df["pred_label_name"] = test_predictions_df["pred_label"].map(id2label)
test_predictions_df["correct"] = test_predictions_df["true_label"] == test_predictions_df["pred_label"]
for class_id, class_name in id2label.items():
    test_predictions_df[f"prob_{class_name.lower()}"] = test_probabilities[:, class_id]

print("Final test metrics from the validation-selected LoRA checkpoint:")
test_metrics


## Сохранение результатов и лучшего adapter checkpoint

Финальная таблица имеет единый формат для последующего сравнения с full fine-tuning. `training_time_sec` в ней относится только к обучению выбранной validation-конфигурации, а не к длительности всего sweep.


In [ ]:
RUN_TABLES_DIR.mkdir(parents=True, exist_ok=True)
BEST_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

FINAL_RESULT_COLUMNS = [
    "model",
    "adaptation",
    "test_macro_f1",
    "test_weighted_f1",
    "test_accuracy",
    "bearish_f1",
    "bullish_f1",
    "neutral_f1",
    "total_params",
    "trainable_params",
    "trainable_share",
    "training_time_sec",
]

selected_parameter_summary = best_run["parameter_summary"]
final_test_row = {
    "model": BASE_MODEL_DISTILBERT,
    "adaptation": "lora",
    "test_macro_f1": test_metrics["macro_f1"],
    "test_weighted_f1": test_metrics["weighted_f1"],
    "test_accuracy": test_metrics["accuracy"],
    "bearish_f1": float(test_classification_report["Bearish"]["f1-score"]),
    "bullish_f1": float(test_classification_report["Bullish"]["f1-score"]),
    "neutral_f1": float(test_classification_report["Neutral"]["f1-score"]),
    "total_params": selected_parameter_summary["total_params"],
    "trainable_params": selected_parameter_summary["trainable_params"],
    "trainable_share": selected_parameter_summary["trainable_share"],
    "training_time_sec": selected_training_time_sec,
}
final_test_results_df = pd.DataFrame([final_test_row], columns=FINAL_RESULT_COLUMNS)
assert list(final_test_results_df.columns) == FINAL_RESULT_COLUMNS
final_test_results_df.to_csv(FINAL_TEST_RESULTS_PATH, index=False)

test_predictions_df.to_csv(
    RUN_TABLES_DIR / "distilbert_lora_best_hparam_test_predictions.csv", index=False
)
test_classification_report_df.to_csv(
    RUN_TABLES_DIR / "distilbert_lora_best_hparam_test_classification_report.csv", index=False
)
test_confusion_matrix_df.to_csv(
    RUN_TABLES_DIR / "distilbert_lora_best_hparam_test_confusion_matrix.csv"
)

best_model.save_pretrained(BEST_ADAPTER_DIR)
tokenizer.save_pretrained(BEST_ADAPTER_DIR)
experiment_config = {
    "dataset_name": DATASET_NAME,
    "model": BASE_MODEL_DISTILBERT,
    "adaptation": "lora",
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "selection_metric": SELECTION_METRIC,
    "selection_split": "validation",
    "test_evaluation_after_validation_selection_only": FINAL_TEST_EVALUATION_ONLY,
    "fixed_grid": LORA_GRID,
    "fixed_training_settings": {
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "target_modules": ["q_lin", "v_lin"],
        "modules_to_save": head_modules_to_save,
    },
    "selected_config": best_config,
    "selected_best_checkpoint": best_run["record"]["best_checkpoint"],
    "selected_training_time_sec": selected_training_time_sec,
    "sweep_results_csv": str(SWEEP_RESULTS_PATH),
    "final_test_results_csv": str(FINAL_TEST_RESULTS_PATH),
}
with EXPERIMENT_CONFIG_PATH.open("w", encoding="utf-8") as config_file:
    json.dump(experiment_config, config_file, indent=2)

print("Saved validation sweep table to:", SWEEP_RESULTS_PATH)
print("Saved final best-model test table to:", FINAL_TEST_RESULTS_PATH)
print("Saved best LoRA adapter and tokenizer to:", BEST_ADAPTER_DIR)
final_test_results_df
